<a href="https://colab.research.google.com/github/mstamatiadou/HAMLET_music_analyzer/blob/main/MtM_Choreography_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate soundfile librosa torch gradio nest_asyncio deepspeed nnAudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.9 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.1-py3-none-any.whl size=1851654 sha256=c16f800d8def5ff7bc8c7e58bd2d70ed6e522b441fa1a52366de8d16a161b4a7
  Stored in directory: /root/.cache/pip/wheels/ac/26/a4/6e4a074ecb7413ef47607c20bef4bcb2180a10330e5e844f92
Successfully built deepspeed


In [ ]:
import asyncio
import gradio as gr
import torch
import librosa
import numpy as np
import json
import torch.nn.functional as F
from transformers import AutoProcessor, ClapModel
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. THE BACKEND ENGINE
# ==========================================
class DynamicTemporalMusicAnalyzer:
    def __init__(self, model_id="laion/clap-htsat-fused"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing Engine on: {self.device}")

        self.model = ClapModel.from_pretrained(model_id).to(self.device)
        self.processor = AutoProcessor.from_pretrained(model_id)

    def analyze_track(self, audio_path, schemes_dict: dict, window_sec=4.0, temperature=3.0):
        target_sr = 48000
        audio_sample, _ = librosa.load(audio_path, sr=target_sr)
        if len(audio_sample.shape) > 1:
            audio_sample = np.mean(audio_sample, axis=0)

        window_samples = int(window_sec * target_sr)
        total_samples = len(audio_sample)
        timeline_signature = []

        for start_idx in range(0, total_samples, window_samples):
            end_idx = start_idx + window_samples
            chunk = audio_sample[start_idx:end_idx]

            if len(chunk) < window_samples / 2:
                continue

            chunk_result = {
                "timestamp_start_sec": round(start_idx / target_sr, 2),
                "timestamp_end_sec": round(end_idx / target_sr, 2),
                "signatures": {}
            }

            for scheme_name, pre_choreo in schemes_dict.items():
                inputs = self.processor(
                    text=pre_choreo,
                    audio=chunk,
                    return_tensors="pt",
                    sampling_rate=target_sr,
                    padding=True
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = self.model(**inputs)

                raw_logits = outputs.logits_per_audio
                probabilities = F.softmax(raw_logits / temperature, dim=-1).cpu().numpy()[0]

                scheme_data = {
                    prompt[:40] + "..." if len(prompt) > 40 else prompt: float(prob)
                    for prompt, prob in zip(pre_choreo, probabilities)
                }
                chunk_result["signatures"][scheme_name] = scheme_data

            timeline_signature.append(chunk_result)

        return timeline_signature

# Initialize the analyzer globally so weights are loaded only once
analyzer = DynamicTemporalMusicAnalyzer()

# ==========================================
# 2. SEABORN PLOT GENERATION FUNCTION
# ==========================================
def generate_timeline_plots(analysis_results):
    """
    Accepts the raw list of dictionaries from the analyzer engine
    and converts it into a single multi-panel Matplotlib figure.
    Configured specifically for Google Colab backends.
    """
    if not analysis_results or len(analysis_results) == 0:
        return None

    all_schemes = list(analysis_results[0]['signatures'].keys())
    num_schemes = len(all_schemes)

    if num_schemes == 0:
        return None

    # Enforce standard plotting aesthetics style
    sns.set_theme(style="whitegrid")

    # Instantiate an independent Figure object explicitly using the Agg backend canvas
    fig, axes = plt.subplots(num_schemes, 1, figsize=(12, 4 * num_schemes), sharex=True)

    # Handle array dimensions defensively for 1D vs Multi-D subplots
    if num_schemes == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for idx, scheme_key in enumerate(all_schemes):
        plot_data = []
        for chunk_info in analysis_results:
            start_time = chunk_info['timestamp_start_sec']
            if scheme_key in chunk_info['signatures']:
                chunk_result = chunk_info['signatures'][scheme_key]
                for element, probability in chunk_result.items():
                    plot_data.append({
                        'start_time_seconds': start_time,
                        'element': element,
                        'probability': probability
                    })

        scheme_df = pd.DataFrame(plot_data)
        if not scheme_df.empty:
            sns.lineplot(
                data=scheme_df, x='start_time_seconds', y='probability',
                hue='element', marker='o', ax=axes[idx]
            )

        clean_title = scheme_key.replace('_', ' ').upper()
        axes[idx].set_title(f'Timeline Signature: {clean_title}', fontsize=11, fontweight='bold', pad=8)
        axes[idx].set_ylabel('Probability')
        axes[idx].set_ylim(0, 1.05)
        axes[idx].legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0., fontsize=9)

    # Set global label parameters on the active figure layout
    axes[-1].set_xlabel('Track Playback Position (Seconds)', fontsize=10, fontweight='bold')
    fig.tight_layout()

    # NOTE: We do NOT call plt.close() here for Colab.
    # Leaving the fig object intact allows Gradio's web canvas to grab the pixels natively.
    return fig

# ==========================================
# 3. THE GUI CONTROLLER LOGIC
# ==========================================
DEFAULT_SCHEMES = {
    "ick_pre_cho_elements": [
        "Rhythm On: clock and sound category, intention where the body submits to the sound",
        "Cosmic origin: intention-based movement originating from the center and stretching to the edge",
        "Rescue oneself: an imperative adaptation movement, changing dynamic through the spine",
        "Ringing and stretching: continuous energetic migration through the wrists, knees, and hips"
    ],
    "traditional_genres": [
        "This is a music track of a classical symphony",
        "This is a music track of funk pop",
        "This is a music track of acoustic folk",
        "This is a music track of heavy rock"
    ]
}

async def process_audio_ui(audio_file_path, custom_genres, window_size, temperature):
    """
    Bridges the front-end interface inputs to backend computations.
    Returns BOTH JSON dictionary and Matplotlib Figure data structures.
    """
    if audio_file_path is None:
        return {"error": "Please upload an audio file."}, None

    # 1. Build runtime schemas
    runtime_schemas = DEFAULT_SCHEMES.copy()

    # 2. Parse custom genres' descriptions
    if custom_genres and custom_genres.strip():
        custom_list = [line.strip() for line in custom_genres.split('\n') if line.strip()]
        if custom_list:
            runtime_schemas["choreographer_custom"] = custom_list

    try:
        # Offload structural audio calculations safely to isolated loop pool
        results = await asyncio.to_thread(
            analyzer.analyze_track,
            audio_path=audio_file_path,
            schemes_dict=runtime_schemas,
            window_sec=window_size,
            temperature=temperature
        )

        # Pass data directly into plotting sub-engine
        fig = generate_timeline_plots(results)

        return results, fig

    except Exception as e:
        return {"error": str(e)}, None

# ==========================================
# 3. GRADIO WEB INTERFACE
# ==========================================
with gr.Blocks(title="HAMLET - Music to Movement Enabler") as demo:
    gr.Markdown("# HAMLET: Music-to-Movement Style Analyzer")
    gr.Markdown("Upload an audio track to analyze its structural intention over time.")

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath", label="Upload Music Track (MP3/WAV)")

            custom_text_input = gr.Textbox(
                lines=5,
                label="Choreographer's Custom Elements (Optional)",
                placeholder="Enter custom descriptive genres here. Each genre shall be in a new line."
            )

            with gr.Accordion("Advanced Configuration", open=False):
                window_slider = gr.Slider(minimum=2.0, maximum=10.0, value=4.0, step=0.5, label="Window Size (seconds)")
                temp_slider = gr.Slider(minimum=1.0, maximum=5.0, value=3.0, step=0.1, label="Softmax Temperature")

            analyze_btn = gr.Button("Analyze Track", variant="primary")

        with gr.Column():
            # Graphical and JSON output elements inside layout interface
            plot_output = gr.Plot(label="Synchronized Visual Progressions")
            json_output = gr.JSON(label="Raw Output Log Data (JSON)")

    # Bind the button directly to our new async controller
    analyze_btn.click(
        fn=process_audio_ui,
        inputs=[audio_input, custom_text_input, window_slider, temp_slider],
        outputs=[json_output, plot_output]
    )

# Launch the UI
demo.launch(share=True, inline=True, debug=True)

Initializing Engine on: cpu


config.json:   0%|          | 0.00/5.42k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/614M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0f4aa9af88aa315bde.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Traditional Music-Genre classification

The following code snipet uses a separate processing pipeline. It loads an audio file, downsamples it to the 24kHz sample rate that **MERT** natively requires, and passes it through the model to print the top genre probabilities.

To get actual genre classifications (like Pop, Classical, Rock, etc.), a fine-tuned version of **MERT** is used, specifically trained for audio genre classification, such as yangwang825/mert-base.

In [4]:
import sys
from transformers import AutoModelForAudioClassification, Wav2Vec2FeatureExtractor, AutoConfig
import torch
import librosa
import numpy as np
import os

try:
    import transformers.deepspeed
except ImportError:
    import transformers
    from types import ModuleType
    ds_mod = ModuleType("deepspeed")
    ds_mod.is_deepspeed_zero3_enabled = lambda: False
    transformers.deepspeed = ds_mod
    sys.modules["transformers.deepspeed"] = ds_mod

def classify_genre_with_mert(audio_path):
    model_name = "yangwang825/mert-base"

    # Φόρτωση config και feature extractor
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForAudioClassification.from_pretrained(model_name, config=config, trust_remote_code=True)

    sampling_rate = 24000
    audio, _ = librosa.load(audio_path, sr=sampling_rate)

    inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)
        logits = outputs.logits

    probs = torch.nn.functional.softmax(logits, dim=-1)

    num_labels = logits.shape[-1]
    k = min(3, num_labels)
    top_probs, top_indices = torch.topk(probs, k=k)

    # Αυτόματη ανάκτηση labels από το configuration του μοντέλου
    id2label = model.config.id2label

    # Προαιρετικό manual mapping μόνο αν το config περιέχει generic LABEL_X
    manual_label_map = {
        "LABEL_0": "Dance/Electronic",
        "LABEL_1": "Pop/Rock",
        "LABEL_2": "Classical",
        "LABEL_3": "Jazz",
        "LABEL_4": "HipHop/Rap"
    }

    results = []
    for i in range(top_indices.shape[1]):
        idx = top_indices[0][i].item()
        # 1. Προσπάθεια ανάκτησης από το id2label του μοντέλου
        label = id2label.get(idx, f"Unknown_{idx}")

        # 2. Αν το label είναι generic (LABEL_X), έλεγχος στο manual_label_map
        friendly_label = manual_label_map.get(label, label)

        score = top_probs[0][i].item()
        results.append((friendly_label, score))

    return results

# Εκτέλεση
target_file = "/content/prettyjohn1-retro-funk-523181.mp3"
if os.path.exists(target_file):
    try:
        predictions = classify_genre_with_mert(target_file)
        print(f"Classification results for {os.path.basename(target_file)}:")
        for label, score in predictions:
            print(f"- {label}: {score:.4f}")
    except Exception as e:
        import traceback
        print(f"An error occurred: {e}")
        traceback.print_exc()
else:
    print("Target file not found.")

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Classification results for prettyjohn1-retro-funk-523181.mp3:
- Dance/Electronic: 0.5191
- Pop/Rock: 0.4809
